In [ ]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent))

import joblib
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import IPython.display as ipd
import soundfile as sf

from art_tts.paths import DATA_DIR

import art_tts.utils_ema.ema_dataset

In [ ]:
def get_gt_audio(filestem, wav_dir):
    original_path = wav_dir / f"{filestem}.wav"
    audio, sr = sf.read(original_path)
    ipd.display(ipd.Audio(audio, rate=sr))

def get_sparc_audio(filestem, hifigan_dir):
    audio_path = hifigan_dir / "sparc" / f"{filestem}.wav"
    audio, sr = sf.read(audio_path)
    ipd.display(ipd.Audio(audio, rate=sr))

def get_model_audios(filestem, version, ckpt_name, hifigan_dir, show_enc=True, suffix=""):
    print(f"\t Enc/dec from {version} {ckpt_name}")
    if show_enc:
        audio_path_enc = hifigan_dir / version / ckpt_name / f"{filestem}_encoder{suffix}.wav"
        audio_enc, sr_enc = sf.read(audio_path_enc)
        ipd.display(ipd.Audio(audio_enc, rate=sr_enc))
    audio_path_dec = hifigan_dir / version / ckpt_name / f"{filestem}_decoder{suffix}.wav"
    audio_dec, sr_dec = sf.read(audio_path_dec)
    ipd.display(ipd.Audio(audio_dec, rate=sr_dec))

Qualitative evaluation of the reconstructed samples from the SPARC-extracted EMA features with the best and worst PCC score (Careful, may be influenced by the quality ofthe linear transfo. The PCC scores are computed on linearly transformed features (from MNGU0 space to concerned speaker) while the reconstruction takes an MNGU0-space input)

We want to assess how powerful the HiFi GAN is and how much it relies on accurate EMA/articulatory predictions.

We also test the sensitivity to pitch and speaker embedding

# LJSpeech-1.1

In [ ]:
from art_tts.utils import parse_filelist

filelist_fp = "resources/filelists/ljspeech/valid_v0.txt"
filelist = parse_filelist(filelist_fp)

In [ ]:
suffix = ""  # "_mean_spk_emb" or "" for own spk_emb
dataset = "LJSpeech-1.1"
version_1 = "v1"
ckpt_name_1 = "grad_3000"
version_2 = "v4"
ckpt_name_2 = "grad_3000"
show_enc = True

hifigan_dir = DATA_DIR / dataset / "hifigan_pred"
wav_dir = DATA_DIR / dataset / "wavs"

for e in filelist[10:16]:
    filestem = e[0].split("/")[-1].split(".")[0]
    print(f"{filestem} sentence : {e[1]}")
    get_gt_audio(filestem, wav_dir)
    get_sparc_audio(filestem, hifigan_dir)
    get_model_audios(filestem, version_1, ckpt_name_1, hifigan_dir, show_enc=show_enc, suffix=suffix)
    get_model_audios(filestem, version_2, ckpt_name_2, hifigan_dir, show_enc=show_enc, suffix=suffix)

# MSPKA_EMA_ita

In [ ]:
suffix = ""  # "_mean_spk_emb" or "" for own spk_emb
dataset = "MSPKA_EMA_ita"
version_1 = "v1"
ckpt_name_1 = "grad_3000"
version_2 = "v4"
ckpt_name_2 = "grad_3000"

version = "v1"
ckpt_name = "grad_5000"
speaker = "cnz"

In [ ]:
suffix = ""  # "_mean_spk_emb" or "" for own spk_emb
dataset = "MSPKA_EMA_ita"
speakers = ["cnz", "olm", "lls"]
min_duration = 5  # seconds
order_by = "pcc_gt_dec" # or "pcc_no_dtw", "pcc_gt_sparc", "pcc_sparc_dec" or "dtw_gt_dec", ...
ascending = False if order_by.startswith("pcc") else True  # get the best at the top
show_enc = False

for speaker in speakers[:3]:
    print(f"Processing speaker: {speaker}")
    hifigan_dir = DATA_DIR / dataset / "arttts" / speaker / "hifigan_pred"
    wav_dir = DATA_DIR / dataset / "src_data" / speaker
    analysis_dir = DATA_DIR / dataset / "arttts" / speaker / "analysis"

    short_df = pd.read_csv(analysis_dir / f"quanti_art_comp_{version}_{ckpt_name}.csv")
    short_df = short_df[short_df["duration"] > min_duration]
    short_df.sort_values(by=order_by, ascending=ascending, inplace=True)
    best_3 = short_df.head(3)
    worst_3 = short_df.tail(3)

    for i, (idx, row) in enumerate(best_3.iterrows()):
        filestem = row["filestem"]
        print(f"Best {i}: {filestem}, metric {order_by}: {row[order_by]:.4f}, duration: {row["duration"]:.2f} seconds")
        get_gt_audio(filestem, wav_dir)
        get_sparc_audio(filestem, hifigan_dir)
        get_model_audios(filestem, version_1, ckpt_name_1, hifigan_dir, show_enc=show_enc, suffix=suffix)
        get_model_audios(filestem, version_2, ckpt_name_2, hifigan_dir, show_enc=show_enc, suffix=suffix)

    for i, (idx, row) in enumerate(worst_3.iterrows()):
        filestem = row["filestem"]
        print(f"Worst {i}: {filestem}, metric {order_by}: {row[order_by]:.4f}, duration: {row['duration']:.2f} seconds")
        get_gt_audio(filestem, wav_dir)
        get_sparc_audio(filestem, hifigan_dir)
        get_model_audios(filestem, version_1, ckpt_name_1, hifigan_dir, show_enc=show_enc, suffix=suffix)
        get_model_audios(filestem, version_2, ckpt_name_2, hifigan_dir, show_enc=show_enc, suffix=suffix)


# pb2007

In [ ]:
suffix = ""  # "_mean_spk_emb" or "" for own spk_emb
dataset = "pb2007"
version_1 = "v1"
ckpt_name_1 = "grad_3000"
version_2 = "v3"
ckpt_name_2 = "grad_3000"

version = "v1"
ckpt_name = "grad_5000"

In [ ]:
suffix = ""  # "_mean_spk_emb" or "" for own spk_emb
dataset = "pb2007"
speakers = ["spk1"]
min_duration = 3  # seconds
order_by = "pcc_gt_dec" # or "pcc_no_dtw", "pcc_gt_sparc", "pcc_sparc_dec" or "dtw_gt_dec", ...
ascending = False if order_by.startswith("pcc") else True  # get the best at the top
show_enc = False

for speaker in speakers[:3]:
    print(f"Processing speaker: {speaker}")
    hifigan_dir = DATA_DIR / dataset / "arttts" / speaker / "hifigan_pred"
    wav_dir = DATA_DIR / dataset / "src_data" / speaker
    analysis_dir = DATA_DIR / dataset / "arttts" / speaker / "analysis"

    short_df = pd.read_csv(analysis_dir / f"quanti_art_comp_{version}_{ckpt_name}.csv")
    short_df = short_df[short_df["sentence_types"] == "sentence"]
    short_df = short_df[short_df["duration"] > min_duration]
    short_df.sort_values(by=order_by, ascending=ascending, inplace=True)
    best_3 = short_df.head(3)
    worst_3 = short_df.tail(3)

    for i, (idx, row) in enumerate(best_3.iterrows()):
        filestem = row["filestem"]
        print(f"Best {i}: {filestem}, metric {order_by}: {row[order_by]:.4f}, duration: {row["duration"]:.2f} seconds")
        get_gt_audio(filestem, wav_dir)
        get_sparc_audio(filestem, hifigan_dir)
        get_model_audios(filestem, version_1, ckpt_name_1, hifigan_dir, show_enc=show_enc, suffix=suffix)
        get_model_audios(filestem, version_2, ckpt_name_2, hifigan_dir, show_enc=show_enc, suffix=suffix)

    for i, (idx, row) in enumerate(worst_3.iterrows()):
        filestem = row["filestem"]
        print(f"Worst {i}: {filestem}, metric {order_by}: {row[order_by]:.4f}, duration: {row['duration']:.2f} seconds")
        get_gt_audio(filestem, wav_dir)
        get_sparc_audio(filestem, hifigan_dir)
        get_model_audios(filestem, version_1, ckpt_name_1, hifigan_dir, show_enc=show_enc, suffix=suffix)
        get_model_audios(filestem, version_2, ckpt_name_2, hifigan_dir, show_enc=show_enc, suffix=suffix)

# mocha_timit

In [ ]:
suffix = ""  # "_mean_spk_emb" or "" for own spk_emb
dataset = "mocha_timit"
version_1 = "v1"
ckpt_name_1 = "grad_3000"
version_2 = "v4"
ckpt_name_2 = "grad_3000"

version = "v1"
ckpt_name = "grad_3000"

In [ ]:
suffix = ""  # "_mean_spk_emb" or "" for own spk_emb
dataset = "mocha_timit"
speakers = ["faet0", "fsew0", "ffes0", "maps0", "mjjn0", "msak0"]
min_duration = 5  # seconds
order_by = "pcc_gt_dec" # or "pcc_no_dtw", "pcc_gt_sparc", "pcc_sparc_dec" or "dtw_gt_dec", ...
ascending = False if order_by.startswith("pcc") else True  # get the best at the top
show_enc = False  # whether to show encoder audio

for speaker in speakers[:6]:
    print(f"Processing speaker: {speaker}")
    hifigan_dir = DATA_DIR / dataset / "arttts" / speaker / "hifigan_pred"
    wav_dir = DATA_DIR / dataset / "src_data" / speaker
    analysis_dir = DATA_DIR / dataset / "arttts" / speaker / "analysis"

    short_df = pd.read_csv(analysis_dir / f"quanti_art_comp_{version}_{ckpt_name}.csv")
    short_df = short_df[short_df["duration"] > min_duration]
    short_df.sort_values(by=order_by, ascending=ascending, inplace=True)
    best_3 = short_df.head(1)
    worst_3 = short_df.tail(1)

    for i, (idx, row) in enumerate(best_3.iterrows()):
        filestem = row["filestem"]
        print(f"Best {i}: {filestem}, metric {order_by}: {row[order_by]:.4f}, duration: {row["duration"]:.2f} seconds")
        get_gt_audio(filestem, wav_dir)
        get_sparc_audio(filestem, hifigan_dir)
        get_model_audios(filestem, version_1, ckpt_name_1, hifigan_dir, show_enc=show_enc, suffix=suffix)
        get_model_audios(filestem, version_2, ckpt_name_2, hifigan_dir, show_enc=show_enc, suffix=suffix)

    for i, (idx, row) in enumerate(worst_3.iterrows()):
        filestem = row["filestem"]
        print(f"Worst {i}: {filestem}, metric {order_by}: {row[order_by]:.4f}, duration: {row['duration']:.2f} seconds")
        get_gt_audio(filestem, wav_dir)
        get_sparc_audio(filestem, hifigan_dir)
        get_model_audios(filestem, version_1, ckpt_name_1, hifigan_dir, show_enc=show_enc, suffix=suffix)
        get_model_audios(filestem, version_2, ckpt_name_2, hifigan_dir, show_enc=show_enc, suffix=suffix)

# MNGU0

In [ ]:
suffix = ""  # "_mean_spk_emb" or "" for own spk_emb
dataset = "MNGU0"
version_1 = "v2"
ckpt_name_1 = "grad_3000"
version_2 = "v4"
ckpt_name_2 = "grad_3000"

version = "v1"
ckpt_name = "grad_5000"

In [ ]:
suffix = ""  # "_mean_spk_emb" or "" for own spk_emb
dataset = "MNGU0"
speakers = ["s1"]
min_duration = 5  # seconds
order_by = "pcc_gt_dec" # or "pcc_no_dtw", "pcc_gt_sparc", "pcc_sparc_dec" or "dtw_gt_dec", ...
ascending = False if order_by.startswith("pcc") else True  # get the best at the top
show_enc = False  # whether to show encoder audio

for speaker in speakers:
    print(f"Processing speaker: {speaker}")
    hifigan_dir = DATA_DIR / dataset / "arttts" / speaker / "hifigan_pred"
    wav_dir = DATA_DIR / dataset / "src_data" / speaker / "wav_16kHz"
    analysis_dir = DATA_DIR / dataset / "arttts" / speaker / "analysis"

    short_df = pd.read_csv(analysis_dir / f"quanti_art_comp_{version}_{ckpt_name}.csv")
    short_df = short_df[short_df["duration"] > min_duration]
    short_df.sort_values(by=order_by, ascending=ascending, inplace=True)
    best_3 = short_df.head(3)
    worst_3 = short_df.tail(3)

    for i, (idx, row) in enumerate(best_3.iterrows()):
        filestem = row["filestem"]
        print(f"Best {i}: {filestem}, metric {order_by}: {row[order_by]:.4f}, duration: {row["duration"]:.2f} seconds")
        get_gt_audio(filestem, wav_dir)
        get_sparc_audio(filestem, hifigan_dir)
        get_model_audios(filestem, version_1, ckpt_name_1, hifigan_dir, show_enc=show_enc, suffix=suffix)
        get_model_audios(filestem, version_2, ckpt_name_2, hifigan_dir, show_enc=show_enc, suffix=suffix)

    for i, (idx, row) in enumerate(worst_3.iterrows()):
        filestem = row["filestem"]
        print(f"Worst {i}: {filestem}, metric {order_by}: {row[order_by]:.4f}, duration: {row['duration']:.2f} seconds")
        get_gt_audio(filestem, wav_dir)
        get_sparc_audio(filestem, hifigan_dir)
        get_model_audios(filestem, version_1, ckpt_name_1, hifigan_dir, show_enc=show_enc, suffix=suffix)
        get_model_audios(filestem, version_2, ckpt_name_2, hifigan_dir, show_enc=show_enc, suffix=suffix)

In [ ]:
dataset = "MNGU0"
speaker = "s1"
src_data_dir = DATA_DIR / dataset / "src_data" / speaker / "ema_basic_data"
emasrc_dir = DATA_DIR / dataset / "arttts" / speaker / "encoded_audio_en" / "emasrc"
spk_emb_dir = DATA_DIR / dataset / "arttts" / speaker / "encoded_audio_en" / "spk_emb"
phnm3_dir = DATA_DIR / dataset / "arttts" / speaker / "phnm3"
reconstructed_dir = DATA_DIR / dataset / "arttts" / speaker / "hifigan_pred"

In [ ]:
from art_tts.utils_dataset.mngu0 import read_mngu0_ema, MNGU0_features
from art_tts.utils import plot_art_14

raw_ema_fp = list(src_data_dir.glob("*"))[522]
raw_ema_fp = raw_ema_fp.parent / "mngu0_s1_1215.ema"
sample_id = raw_ema_fp.stem
ema_data, nonan = read_mngu0_ema(raw_ema_fp)
sparc_ema = np.load(emasrc_dir / f"{sample_id}.npy")[:,:12]
phnm3 = np.load(phnm3_dir / f"{sample_id}_phnm3.npy")

ema_data_14 = np.pad(ema_data, ((0, 0), (0, 2)), mode='constant', constant_values=0)
ema_data_14 = (ema_data_14 - ema_data_14.mean(axis=0)) / ema_data_14.std(axis=0)
ema_data_14 = ema_data_14[::4,:]
sparc_ema_14 = np.pad(sparc_ema, ((0, 0), (0, 2)), mode='constant', constant_values=0)

fig, _ = plot_art_14([ema_data_14.T,
                       sparc_ema_14.T],
                       phnm3=phnm3,
                       title=f"Sample {sample_id} - MNGU0",
                       figsize=(16, 10),)
fig

In [ ]:
from scipy.stats import pearsonr

suffix = "_mean_spk_emb"  # "_mean_spk_emb" or "" for own spk_emb
filestems = []
pccs = []
durations = []
for raw_ema_fp in list(src_data_dir.glob("*.ema")):
    sample_id = raw_ema_fp.stem
    ema_data, nonan = read_mngu0_ema(raw_ema_fp)
    if nonan: #and ema_data.shape[0] / 4 > 150:
        filestems.append(sample_id)
        durations.append(ema_data.shape[0] / 200) # Original data 200 Hz
        ema_data = (ema_data - ema_data.mean(axis=0)) / ema_data.std(axis=0)
        ema_data = ema_data[::4,:]  # TO 50 Hz
        sparc_ema = np.load(emasrc_dir / f"{sample_id}.npy")[:,:12]
        #phnm3 = np.load(phnm3_dir / f"{sample_id}_phnm3.npy")
        pcc, _ = pearsonr(sparc_ema, ema_data[:sparc_ema.shape[0]])
        pccs.append(pcc)
pccs = np.array(pccs)

summary_df = pd.DataFrame({
    "filestem": filestems,
    "pcc": pccs.mean(axis=1),
    "duration": durations
})
summary_df = summary_df.sort_values(by="pcc", ascending=False)

# Keep the top 3 and bottom 3 entries based on PCC scores
best_3 = summary_df.head(3)
worst_3 = summary_df.tail(3)

# get their sparc reconstruction
reconstructed_dir = DATA_DIR / dataset / "arttts" / speaker / "hifigan_pred"

for i, row in best_3.iterrows():
    filestem = row["filestem"]
    pcc = row["pcc"]
    duration = row["duration"]
    print(f"Best {i}: {filestem}, PCC: {pcc:.4f}, duration: {duration:.2f} seconds")
    audio_path = reconstructed_dir / "sparc" / f"{filestem}.wav"
    audio_path_enc = reconstructed_dir / "v1_/grad_4750" / f"{filestem}_encoder{suffix}.wav"
    audio_path_dec = reconstructed_dir / "v1_/grad_4750" / f"{filestem}_decoder{suffix}.wav"
    audio, sr = sf.read(audio_path)
    ipd.display(ipd.Audio(audio, rate=sr))
    audio_enc, sr_enc = sf.read(audio_path_enc)
    ipd.display(ipd.Audio(audio_enc, rate=sr_enc))
    audio_dec, sr_dec = sf.read(audio_path_dec)
    ipd.display(ipd.Audio(audio_dec, rate=sr_dec))

for i, row in worst_3.iterrows():
    filestem = row["filestem"]
    pcc = row["pcc"]
    duration = row["duration"]
    print(f"Worst {i}: {filestem}, PCC: {pcc:.4f}, duration: {duration:.2f} seconds")
    audio_path = reconstructed_dir / "sparc" / f"{filestem}.wav"
    audio_path_enc = reconstructed_dir / "v1_/grad_4750" / f"{filestem}_encoder{suffix}.wav"
    audio_path_dec = reconstructed_dir / "v1_/grad_4750" / f"{filestem}_decoder{suffix}.wav"
    audio, sr = sf.read(audio_path)
    ipd.display(ipd.Audio(audio, rate=sr))
    audio_enc, sr_enc = sf.read(audio_path_enc)
    ipd.display(ipd.Audio(audio_enc, rate=sr_enc))
    audio_dec, sr_dec = sf.read(audio_path_dec)
    ipd.display(ipd.Audio(audio_dec, rate=sr_dec))

In [ ]:
# Calculate mean and standard deviation for each feature
means = pccs.mean(axis=0)
stds = pccs.std(axis=0)
print(means.mean())
print(stds.mean())

# Generate the bar plot
plt.figure(figsize=(6, 4))
plt.grid()
plt.bar(MNGU0_features, means, yerr=stds, capsize=5, color='skyblue', edgecolor='black')
plt.xticks(rotation=45)
plt.xlabel("Features")
plt.ylabel("PCC")
plt.ylim((0.5,1))
plt.title("PCC between SPARC and EMA features with Standard Deviation")
plt.tight_layout()
plt.show()

In [ ]:
summary_df.plot(x="duration", y="pcc", kind="scatter");

In [ ]:
sum_err2 = np.zeros((1000, 12))
counts = np.zeros((1000,1))

In [ ]:
max_frames = 0
for raw_ema_fp in list(src_data_dir.glob("*.ema")):
    sample_id = raw_ema_fp.stem
    ema_data, nonan = read_mngu0_ema(raw_ema_fp)
    if nonan: #and ema_data.shape[0] / 4 > 150:
        filestems.append(sample_id)
        durations.append(ema_data.shape[0] / 200) # Original data 200 Hz
        ema_data = (ema_data - ema_data.mean(axis=0)) / ema_data.std(axis=0)
        ema_data = ema_data[::4,:]  # TO 50 Hz
        sparc_ema = np.load(emasrc_dir / f"{sample_id}.npy")[:,:12]
        err2 = (sparc_ema - ema_data[:sparc_ema.shape[0]])**2
        nframes = err2.shape[0]
        sum_err2[:nframes] += err2
        counts[:nframes] += 1
        if nframes > max_frames:
            max_frames = nframes
counts[max_frames:] = 1
mean_err2 = np.sqrt(sum_err2 / counts)

In [ ]:
plt.plot(np.arange(1000) / 50,mean_err2)
#plt.plot(np.arange(1000) / 50,counts, label="mean")

# MOS

## UTMOS

## ljspeech

In [ ]:
import os
import shutil
import random


dataset = "LJSpeech-1.1"
dest_dir = DATA_DIR / f"{dataset}_listen"
scores_csv = dest_dir / "scores.csv"
utmos_csv = dest_dir / "utmos.csv"
scores_df = pd.read_csv(scores_csv)
utmos_df = pd.read_csv(utmos_csv, names=["uuid", "utmos_score"])
scores_df["uuid"] = scores_df.apply(lambda row: f"{row["filestem"]}_{row["suffix"]}", axis=1)
scores_df = scores_df.merge(utmos_df, on="uuid", how="left")
scores_df["utmos_score"] = scores_df["utmos_score"].astype(np.float32)
scores_df = scores_df[["uuid", "source", "utmos_score"]]
scores_df.groupby("source").mean("utmos_score").sort_values(by="utmos_score", ascending=False)

## MNGU0

In [ ]:
import os
import shutil
import random


dataset = "MNGU0"
dest_dir = DATA_DIR / f"{dataset}_listen"
scores_csv = dest_dir / "scores.csv"
utmos_csv = dest_dir / "utmos.csv"
scores_df = pd.read_csv(scores_csv)
utmos_df = pd.read_csv(utmos_csv, names=["uuid", "utmos_score"])
scores_df["uuid"] = scores_df.apply(lambda row: f"{row["filestem"]}_{row["suffix"]}", axis=1)
scores_df = scores_df.merge(utmos_df, on="uuid", how="left")
scores_df["utmos_score"] = scores_df["utmos_score"].astype(np.float32)
scores_df = scores_df[["uuid", "source", "utmos_score"]]
scores_df.groupby("source").mean("utmos_score").sort_values(by="utmos_score", ascending=False)

## MSPKA

In [ ]:
import os
import shutil
import random


dataset = "MSPKA_EMA_ita"
dest_dir = DATA_DIR / f"{dataset}_listen"
scores_csv = dest_dir / "scores.csv"
utmos_csv = dest_dir / "utmos.csv"
scores_df = pd.read_csv(scores_csv)
utmos_df = pd.read_csv(utmos_csv, names=["uuid", "utmos_score"])
scores_df["uuid"] = scores_df.apply(lambda row: f"{row["filestem"]}_{row["suffix"]}", axis=1)
scores_df = scores_df.merge(utmos_df, on="uuid", how="left")
scores_df["utmos_score"] = scores_df["utmos_score"].astype(np.float32)
scores_df = scores_df[["uuid", "source", "utmos_score"]]
scores_df.groupby("source").mean("utmos_score").sort_values(by="utmos_score", ascending=False)

## Mocha

In [ ]:
import os
import shutil
import random


dataset = "mocha_timit"
dest_dir = DATA_DIR / f"{dataset}_listen"
scores_csv = dest_dir / "scores.csv"
utmos_csv = dest_dir / "utmos.csv"
scores_df = pd.read_csv(scores_csv)
utmos_df = pd.read_csv(utmos_csv, names=["uuid", "utmos_score"])
scores_df["uuid"] = scores_df.apply(lambda row: f"{row["filestem"]}_{row["suffix"]}", axis=1)
scores_df = scores_df.merge(utmos_df, on="uuid", how="left")
scores_df["utmos_score"] = scores_df["utmos_score"].astype(np.float32)
scores_df = scores_df[["uuid", "source", "utmos_score"]]
scores_df.groupby("source").mean("utmos_score").sort_values(by="utmos_score", ascending=False)

In [ ]:
versions = ["src", "sparc", "v1_1", "v4"]
datasets = ["LJSpeech-1.1", "MSPKA_EMA_ita", "mocha_timit", "MNGU0"]
utmos_summary = None
for dataset in ["LJSpeech-1.1", "MSPKA_EMA_ita", "mocha_timit", "MNGU0"]:
    dest_dir = DATA_DIR / f"{dataset}_listen"
    scores_csv = dest_dir / "scores.csv"
    utmos_csv = dest_dir / "utmos.csv"
    scores_df = pd.read_csv(scores_csv)
    utmos_df = pd.read_csv(utmos_csv, names=["uuid", "utmos_score"])
    scores_df["uuid"] = scores_df.apply(lambda row: f"{row["filestem"]}_{row["suffix"]}", axis=1)
    scores_df = scores_df.merge(utmos_df, on="uuid", how="left")
    scores_df["utmos_score"] = scores_df["utmos_score"].astype(np.float32)
    scores_df = scores_df[["uuid", "source", "utmos_score"]]
    version_scores = scores_df.groupby("source").mean("utmos_score").sort_values(by="utmos_score", ascending=False)
    version_scores.rename(columns={"utmos_score": f"{dataset}"}, inplace=True)
    version_scores.reset_index(inplace=True)
    if utmos_summary is None:
        utmos_summary = version_scores
    else:
        utmos_summary = utmos_summary.merge(version_scores, how="left", on="source")
utmos_summary = utmos_summary.set_index("source").transpose()
utmos_summary.rename(index={"source": "dataset"}, inplace=True)
utmos_summary

In [ ]:
rearranged_utmos = utmos_summary[["src", "sparc", "v1_1", "v4"]].transpose()[["LJSpeech-1.1", "mocha_timit", "MNGU0", "MSPKA_EMA_ita"]]
rearranged_utmos

In [ ]:
for source in rearranged_utmos.index:
    plt.plot(rearranged_utmos.columns, rearranged_utmos.loc[source], label=source, marker='o')
plt.xlabel("Source")
plt.ylabel("UTMOS Score")
plt.title("UTMOS Scores by Source for Different Datasets")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

# check v1 as well (v2 too)

In [ ]:
import shutil
import os

datasets = ["LJSpeech-1.1", "MSPKA_EMA_ita", "mocha_timit", "MNGU0"]
for dataset in datasets:
    speaker = ""
    ckpt_name = "grad_3000"
    artts_folder = "arttts"
    if dataset == "LJSpeech-1.1":
        artts_folder = ""
        
    filestems = []
    # Define the source directory, destination directory, and list of filestems
    src_dir = DATA_DIR / f"{dataset}_listen"
    dest_dir = DATA_DIR / f"{dataset}_listen_v2"
    os.makedirs(dest_dir, exist_ok=True)

    scores_csv = src_dir / "scores.csv"
    scores_df = pd.read_csv(scores_csv)
    filestems = scores_df["filestem"].unique()

    for filestem in filestems:
        if dataset == "mocha_timit":
            speaker = filestem.split("_")[0]
        elif dataset == "MSPKA_EMA_ita":
            speaker = filestem.split("_")[0]
        elif dataset == "MNGU0":
            speaker = "s1"
        v2_dir = DATA_DIR / dataset / artts_folder / speaker  / "hifigan_pred" / "v2" / ckpt_name
        v2_file = v2_dir / f"{filestem}_decoder.wav"
        dest_v2_file = dest_dir / f"{filestem}.wav"
        #shutil.copy(v2_file, dest_v2_file)

In [ ]:
versions = ["src", "sparc", "v1_1", "v4"]
add_versions = ["v1", "v2"]
datasets = ["LJSpeech-1.1", "MSPKA_EMA_ita", "mocha_timit", "MNGU0"]
utmos_summary = None
utmos_add_versions = {}
for dataset in ["LJSpeech-1.1", "MSPKA_EMA_ita", "mocha_timit", "MNGU0"]:
    dest_dir = DATA_DIR / f"{dataset}_listen"
    scores_csv = dest_dir / "scores.csv"
    utmos_csv = dest_dir / "utmos.csv"
    scores_df = pd.read_csv(scores_csv)
    utmos_df = pd.read_csv(utmos_csv, names=["uuid", "utmos_score"])
    scores_df["uuid"] = scores_df.apply(lambda row: f"{row["filestem"]}_{row["suffix"]}", axis=1)
    scores_df = scores_df.merge(utmos_df, on="uuid", how="left")
    scores_df["utmos_score"] = scores_df["utmos_score"].astype(np.float32)
    scores_df = scores_df[["uuid", "source", "utmos_score"]]
    version_scores = scores_df.groupby("source").mean("utmos_score").sort_values(by="utmos_score", ascending=False)
    version_scores.rename(columns={"utmos_score": f"{dataset}"}, inplace=True)
    version_scores.reset_index(inplace=True)
    if utmos_summary is None:
        utmos_summary = version_scores
    else:
        utmos_summary = utmos_summary.merge(version_scores, how="left", on="source")
    
    for v in add_versions:
        dest_dir_v = DATA_DIR / f"{dataset}_listen_{v}"
        utmos_csv_v = dest_dir_v / "utmos.csv"
        utmos_df_v = pd.read_csv(utmos_csv_v, names=["uuid", "utmos_score"])
        if v not in utmos_add_versions:
            utmos_add_versions[v] = []
        utmos_add_versions[v].append(utmos_df_v["utmos_score"].mean())

utmos_summary = utmos_summary.set_index("source").transpose()
utmos_summary.rename(index={"source": "dataset"}, inplace=True)
for v in add_versions:
    utmos_summary[v] = utmos_add_versions[v]
utmos_summary

In [ ]:
rearranged_utmos = utmos_summary[versions + add_versions].transpose()[["LJSpeech-1.1", "mocha_timit", "MNGU0", "MSPKA_EMA_ita"]]
rearranged_utmos

In [ ]:
for source in rearranged_utmos.index:
    plt.plot(rearranged_utmos.columns, rearranged_utmos.loc[source], label=source, marker='o')
plt.xlabel("Source")
plt.ylabel("UTMOS Score")
plt.title("UTMOS Scores by Source for Different Datasets")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
angelo_mos = [[5.00,  3.89, 4.67, 3.67],
        [3.56,  3.56, 4.33, 3.78],
        [3.11,  2.89, 2.44, 2.33],
        [2.56,  3.22, 3.00, 2.00]]
for i, e in enumerate(angelo_mos):
    plt.plot(["LJSpeech-1.1", "mocha_timit", "MNGU0", "MSPKA_EMA_ita"], e, label=f"{versions[i]}", linestyle='--', marker='x')
plt.legend()

# Compare Angelo vs UTMOS scores

In [ ]:
from scipy.stats import pearsonr

versions = ["src", "sparc", "v1_1", "v4"]
datasets = ["LJSpeech-1.1", "MSPKA_EMA_ita", "mocha_timit", "MNGU0"]
utmos_versions = {}
angelo_versions = {}
for dataset in ["LJSpeech-1.1", "MSPKA_EMA_ita", "mocha_timit", "MNGU0"]:
    dest_dir = DATA_DIR / f"{dataset}_listen"
    scores_csv = dest_dir / "scores.csv"
    utmos_csv = dest_dir / "utmos.csv"
    scores_df = pd.read_csv(scores_csv)
    utmos_df = pd.read_csv(utmos_csv, names=["uuid", "utmos_score"])
    scores_df["uuid"] = scores_df.apply(lambda row: f"{row["filestem"]}_{row["suffix"]}", axis=1)
    scores_df = scores_df.merge(utmos_df, on="uuid", how="left")
    scores_df["utmos_score"] = scores_df["utmos_score"].astype(np.float32)
    scores_df = scores_df[["uuid", "source", "utmos_score", "angelo"]]
    for v in versions:
        short_df = scores_df[scores_df["source"] == v]
        if v not in utmos_versions:
            utmos_versions[v] = []
            angelo_versions[v] = []
        utmos_versions[v] += list(short_df["utmos_score"])
        angelo_versions[v] += list(short_df["angelo"])
    
for k in utmos_versions.keys():
    pcc, pvalues = pearsonr(utmos_versions[k], angelo_versions[k])
    print(f"Version {k} - UTMOS vs Angelo PCC: {pcc:.4f}, p-value: {pvalues:.4f}")

# Compare v1 v1_1 whole dataset.

In [ ]:
versions = ["src", "sparc", "v1", "v1_1", "v4_", "v4", "v2"]
versions = ["src", "sparc", "v1", "v1_1", "v4_", "v2"]
datasets = ["LJSpeech-1.1", "mocha_timit", "MNGU0", "MSPKA_EMA_ita"]
#dataset = "LJSpeech-1.1"
#versions = ["v1", "v1_1"]

dataset_means = {}
for dataset in datasets:
    for version in versions:
        utmos_csv = DATA_DIR / f"{dataset}/UTMOS_data/decoder_files_{version}/utmos.csv"
        utmos_df = pd.read_csv(utmos_csv, names=["uuid", "utmos_score"])
        mean = utmos_df["utmos_score"].mean()
        if dataset not in dataset_means:
            dataset_means[dataset] = {}
        dataset_means[dataset][version] = mean

UTMOS_full = pd.DataFrame(dataset_means)
#UTMOS_full.rename(index={"v4_": "v4_1.0",
#                         "v4" : "v4_1.25"}, inplace=True)
UTMOS_full.rename(index={"v4_": "v4",
                         "src":"original"}, inplace=True)
# round to 2 decimal places
UTMOS_full = UTMOS_full.round(2)
UTMOS_full

In [ ]:
ax = UTMOS_full.transpose().plot(kind='bar', figsize=(8, 4))
ax.grid(axis="y")
ax.legend(title="Versions", bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_title("Mean UTMOS Scores by Dataset and Version")
ax.set_ylabel("Mean UTMOS Score")
ax.set_xlabel("Dataset")
plt.tight_layout()
;

# Compare v2_full at different ckpts

In [ ]:
from art_tts.paths import DATA_DIR
import pandas as pd

#versions = ["src", "sparc", "v1", "v1_1", "v4_", "v4", "v2"]
ckpts = ["grad_2000", "grad_4000", "grad_6000", "grad_8000", "grad_10000"]
datasets = ["LJSpeech-1.1", "mocha_timit", "MNGU0", "MSPKA_EMA_ita"]
version = "v2_full"

dataset_means = {}
for dataset in datasets:
    for ckpt in ckpts:
        utmos_csv = DATA_DIR / f"{dataset}/UTMOS_data/decoder_files_{version}/{ckpt}/utmos.csv"
        utmos_df = pd.read_csv(utmos_csv, names=["uuid", "utmos_score"])
        mean = utmos_df["utmos_score"].mean()
        if dataset not in dataset_means:
            dataset_means[dataset] = {}
        dataset_means[dataset][ckpt] = mean

UTMOS_full = pd.DataFrame(dataset_means)
UTMOS_full

In [ ]:
UTMOS_full.transpose().plot(kind='bar', figsize=(8, 4));

# create full samples dir

In [ ]:
import numpy as np
import soundfile as sf
import torchaudio



def change_audio_speed(input_path, output_path, speed=1.25):
    """
    Change the playback speed of an audio file.

    Parameters:
    - input_path: Path to the input audio file.
    - output_path: Path to save the modified audio file.
    - speed: Speed factor (e.g., 1.5 for 1.5x speed).
    """
    waveform, sr = torchaudio.load(input_path)
    effects = [["tempo", str(speed)]]  # faster, pitch preserved
    waveform, sr = torchaudio.sox_effects.apply_effects_tensor(waveform, sr, effects)
    torchaudio.save(output_path, waveform, sr)

In [ ]:
#dataset = "LJSpeech-1.1"
#valid_dir = DATA_DIR / dataset / "UTMOS_data" / "decoder_files_v1"
#valid_fn = [str(Path(f).name).replace("_decoder", "") for f in valid_dir.glob("*.wav")]
#
#sparc_dir = DATA_DIR / dataset / "hifigan_pred" / "sparc"
#src_dir = DATA_DIR / dataset / "wavs"
#v4_dir = DATA_DIR / dataset / "hifigan_pred" / "v4" / "grad_3000"
#
#sparc_utmos_dir = DATA_DIR / dataset / "UTMOS_data" / "decoder_files_sparc"
#sparc_utmos_dir.mkdir(parents=True, exist_ok=True)
#src_utmos_dir = DATA_DIR / dataset / "UTMOS_data" / "decoder_files_src"
#src_utmos_dir.mkdir(parents=True, exist_ok=True)
#v4_utmos_dir = DATA_DIR / dataset / "UTMOS_data" / "decoder_files_v4"
#
#for fn in valid_fn:
#    sparc_audio_path = sparc_dir / fn
#    src_audio_path = src_dir / fn
#    v4_audio_path = v4_dir / fn.replace(".wav", "_decoder.wav")
#    
#    if sparc_audio_path.exists():
#        shutil.copy(sparc_audio_path, sparc_utmos_dir / fn)
#    else:
#        print(f"Sparc audio not found for {fn}")
#    
#    if src_audio_path.exists():
#        shutil.copy(src_audio_path, src_utmos_dir / fn)
#    else:
#        print(f"Source audio not found for {fn}")
#    
#    if v4_audio_path.exists():
#        change_audio_speed(str(v4_audio_path), str(v4_utmos_dir / fn), speed=1.25)
#    else:
#        print(f"V4 audio not found for {fn}")

In [ ]:
#datasets = ["MSPKA_EMA_ita", "mocha_timit", "MNGU0"]
#for dataset in datasets:
#    speaker = ""
#    ckpt_name = "grad_3000"
#    artts_folder = "arttts"
#
#    valid_dir = DATA_DIR / dataset / "UTMOS_data" / "decoder_files_v1"
#    filestems = [f.stem.replace('_decoder', '') for f in valid_dir.glob("*.wav")]
#    
#    sparc_utmos_dir = DATA_DIR / dataset / "UTMOS_data" / "decoder_files_sparc"
#    sparc_utmos_dir.mkdir(parents=True, exist_ok=True)
#    src_utmos_dir = DATA_DIR / dataset / "UTMOS_data" / "decoder_files_src"
#    src_utmos_dir.mkdir(parents=True, exist_ok=True)
#    v4_utmos_dir = DATA_DIR / dataset / "UTMOS_data" / "decoder_files_v4"
#    v4_utmos_dir.mkdir(parents=True, exist_ok=True)
#
#
#    for filestem in filestems:
#        if dataset == "mocha_timit":
#            speaker = filestem.split("_")[0]
#            src_dir = DATA_DIR / dataset / "src_data" / speaker
#        elif dataset == "MSPKA_EMA_ita":
#            speaker = filestem.split("_")[0]
#            src_dir = DATA_DIR / dataset / "src_data" / speaker
#        elif dataset == "MNGU0":
#            speaker = "s1"
#            src_dir = DATA_DIR / dataset / "src_data" / speaker / "wav_16kHz"
#        v4_dir = DATA_DIR / dataset / artts_folder / speaker / "hifigan_pred" / "v4" / ckpt_name
#        sparc_dir = DATA_DIR / dataset / artts_folder / speaker / "hifigan_pred" / "sparc"
#        v4_audio_path = v4_dir / f"{filestem}_decoder.wav"
#        sparc_audio_path = sparc_dir / f"{filestem}.wav"
#        src_audio_path = src_dir / f"{filestem}.wav"
#
#        if sparc_audio_path.exists():
#            shutil.copy(sparc_audio_path, sparc_utmos_dir / f"{filestem}.wav")
#        else:
#            print(f"Sparc audio not found for {f"{filestem}.wav"}")
#        
#        if src_audio_path.exists():
#            shutil.copy(src_audio_path, src_utmos_dir / f"{filestem}.wav")
#        else:
#            print(f"Source audio not found for {f"{filestem}.wav"}")
#        
#        if v4_audio_path.exists():
#            change_audio_speed(str(v4_audio_path), str(v4_utmos_dir / f"{filestem}_decoder.wav"), speed=1.25)
#        else:
#            print(f"V4 audio not found for {f"{filestem}_decoder.wav"}")
